# Task 4 — Data Augmentation and CNN Optimization
### Sentinel-2 Vegetation Detection

**Goal:** Test data augmentation variations on the CNN from Task 3 and record which improves generalization. We start with Horizontal Flip (HFlip).

This notebook follows the same data pipeline as Task 2/3 (NDVI > 0.3 = vegetation).

In [ ]:
# Install required libraries (run once)
import sys
!{sys.executable} -m pip install torch torchvision albumentations Pillow numpy matplotlib scikit-learn --quiet

In [ ]:
import os, random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import albumentations as A

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "Device:", DEVICE)

## 1 · Configuration
Same paths as Task 3. Change these if your data is elsewhere.

In [ ]:
ROUTE_SAMPLES = "data/samples"   # folder with *_img_*.tiff
ROUTE_LABELS = "data/labels"     # folder with *_ndvi_*.tiff

NDVI_THRESHOLD = 0.3
PATCH_SIZE = 32
STRIDE = 8
BATCH_SIZE = 128
EPOCHS = 30
LEARNING_RATE = 1e-3

print("Config loaded")

## 2 · Data Loading
Load RGB and NDVI exactly like Task 3, normalize RGB to [0,1] and NDVI to [-1,1].

In [ ]:
from PIL import Image

def load_pair(img_path, ndvi_path):
    with Image.open(img_path) as im:
        rgb = np.array(im).astype(np.float32)
    rgb = rgb / rgb.max() if rgb.max()>0 else rgb

    with Image.open(ndvi_path) as nm:
        ndvi = np.array(nm).astype(np.float32)
    if ndvi.max() > 1 or ndvi.min() < -1:
        ndvi = (ndvi - ndvi.min())/(ndvi.max()-ndvi.min())*2 - 1

    mask = (ndvi > NDVI_THRESHOLD).astype(np.int64)
    return rgb, ndvi, mask

def discover_pairs(samples_dir, labels_dir):
    files = [f for f in os.listdir(samples_dir) if f.endswith(".tiff")]
    paired = [f for f in files if os.path.exists(os.path.join(labels_dir, f.replace("_img_","_ndvi_")))]
    return [(os.path.join(samples_dir,f), os.path.join(labels_dir,f.replace("_img_","_ndvi_"))) for f in sorted(paired)]

pairs = discover_pairs(ROUTE_SAMPLES, ROUTE_LABELS)
print(f"Found {len(pairs)} pairs")

all_rgb, all_ndvi, all_mask = [], [], []
for img_p, ndvi_p in pairs:
    rgb, ndvi, mask = load_pair(img_p, ndvi_p)
    all_rgb.append(rgb); all_ndvi.append(ndvi); all_mask.append(mask)
print("Data loaded")

## 2.1 · Visualize RGB, NDVI and Mask
Like Task 2/3, we display a sample to confirm NDVI thresholding.


In [ ]:
# visualize first 3 images
def show_sample(idx=0):
    rgb = all_rgb[idx]
    ndvi = all_ndvi[idx]
    mask = all_mask[idx]
    
    fig, axes = plt.subplots(1,3, figsize=(15,5))
    axes[0].imshow(np.clip(rgb,0,1))
    axes[0].set_title(f'RGB Sample {idx}')
    axes[0].axis('off')
    
    im = axes[1].imshow(ndvi, cmap='RdYlGn', vmin=-1, vmax=1)
    axes[1].set_title('NDVI')
    axes[1].axis('off')
    plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
    
    axes[2].imshow(mask, cmap='Greens', vmin=0, vmax=1)
    axes[2].set_title(f'Vegetation Mask > {NDVI_THRESHOLD}')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

for i in range(min(3, len(all_rgb))):
    show_sample(i)


## 3 · Patch Dataset with Augmentation
We extract 32x32 patches. The label is the majority class in the centre 8x8 region. 
This cell defines the augmentation for **HFlip** only.

In [ ]:
# --- Task 4 Augmentations (Albumentations) - FIXED with Resize ---
# PATCH_SIZE is defined in config (32)

# 1. Random Rotation
AUG_ROTATION = A.Compose([
    A.Rotate(limit=30, p=1.0, border_mode=0),
    A.Resize(PATCH_SIZE, PATCH_SIZE),
], additional_targets={'mask':'mask'})

# 2. Random Brightness
AUG_BRIGHTNESS = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0, p=1.0),
    A.Resize(PATCH_SIZE, PATCH_SIZE),
], additional_targets={'mask':'mask'})

# 3. Random Shifts (45%) - using Affine
AUG_SHIFT = A.Compose([
    A.Affine(
        translate_percent={'x': (-0.45, 0.45), 'y': (-0.45, 0.45)},
        scale=1.0,
        rotate=0,
        p=1.0,
        border_mode=0
    ),
    A.Resize(PATCH_SIZE, PATCH_SIZE),
], additional_targets={'mask':'mask'})

# 4. Random Flips
AUG_FLIPS = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Resize(PATCH_SIZE, PATCH_SIZE),
], additional_targets={'mask':'mask'})

# 5. Random Zoom (most helpful)
AUG_ZOOM = A.Compose([
    A.RandomScale(scale_limit=0.4, p=1.0),  # 0.6 to 1.4
    A.Resize(PATCH_SIZE, PATCH_SIZE),       # force back to 32x32 for batching
], additional_targets={'mask':'mask'})

# Legacy names for compatibility
AUG_HFLIP = A.Compose([A.HorizontalFlip(p=1.0), A.Resize(PATCH_SIZE, PATCH_SIZE)], additional_targets={'mask':'mask'})
AUG_VFLIP = A.Compose([A.VerticalFlip(p=1.0), A.Resize(PATCH_SIZE, PATCH_SIZE)], additional_targets={'mask':'mask'})
AUG_ROT90 = A.Compose([A.RandomRotate90(p=1.0), A.Resize(PATCH_SIZE, PATCH_SIZE)], additional_targets={'mask':'mask'})
AUG_PHOTOMETRIC = A.Compose([
    A.RandomRotate90(p=1.0),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.Resize(PATCH_SIZE, PATCH_SIZE),
], additional_targets={'mask':'mask'})

class PatchDatasetAug(Dataset):
    def __init__(self, rgb_list, mask_list, patch_size, stride, transform=None):
        self.samples = []
        half = patch_size // 2
        for rgb, mask in zip(rgb_list, mask_list):
            H,W = mask.shape
            for y in range(half, H-half, stride):
                for x in range(half, W-half, stride):
                    patch_rgb = rgb[y-half:y+half, x-half:x+half, :]
                    patch_msk = mask[y-half:y+half, x-half:x+half]
                    centre = patch_msk[12:20, 12:20]
                    label = int(centre.mean() > 0.5)
                    self.samples.append((patch_rgb, patch_msk, label))
        self.transform = transform

    def __len__(self): return len(self.samples)
    
    def __getitem__(self, idx):
        rgb, msk, label = self.samples[idx]
        if self.transform:
            aug = self.transform(image=(rgb*255).astype(np.uint8), mask=msk)
            rgb = aug['image'].astype(np.float32) / 255.0
        rgb = torch.tensor(rgb.transpose(2,0,1), dtype=torch.float32)
        return rgb, torch.tensor(label, dtype=torch.long)


In [ ]:
# Dictionary of all augmentations for Task 4
AUGMENTATIONS = {
    'random_rotation': AUG_ROTATION,
    'random_brightness': AUG_BRIGHTNESS,
    'random_shift': AUG_SHIFT,
    'random_flips': AUG_FLIPS,
    'random_zoom': AUG_ZOOM,
    # previous runs
    'hflip': AUG_HFLIP,
    'vflip': AUG_VFLIP,
    'rotation90': AUG_ROT90,
    'photometric': AUG_PHOTOMETRIC,
}
print('Available:', list(AUGMENTATIONS.keys()))


### Visualize Patches with HFlip and other popular arguments. 
Compare original patch vs horizontally flipped version.


In [ ]:
# create small datasets for visualization
vis_orig = PatchDatasetAug(all_rgb, all_mask, PATCH_SIZE, STRIDE, transform=None)
#vis_flip = PatchDatasetAug(all_rgb, all_mask, PATCH_SIZE, STRIDE, transform=AUG_VFLIP)
#vis_flip = PatchDatasetAug(all_rgb, all_mask, PATCH_SIZE, STRIDE, transform=AUG_ROT)
#vis_flip = PatchDatasetAug(all_rgb, all_mask, PATCH_SIZE, STRIDE, transform=AUG_PHOTO)
#vis_flip = PatchDatasetAug(all_rgb, all_mask, PATCH_SIZE, STRIDE, transform=AUG_ZOOM)
# New Arguments 
vis_flip = PatchDatasetAug(all_rgb, all_mask, PATCH_SIZE, STRIDE, transform=AUG_SHIFT)

def plot_patches(n=6):
    fig, axes = plt.subplots(2, n, figsize=(n*2.5, 5))
    for i in range(n):
        # original
        rgb_o, _ = vis_orig[i]
        img_o = rgb_o.permute(1,2,0).numpy()
        axes[0,i].imshow(np.clip(img_o,0,1))
        axes[0,i].set_title('Original')
        axes[0,i].axis('off')
        
        # flipped
        rgb_f, _ = vis_flip[i]
        img_f = rgb_f.permute(1,2,0).numpy()
        axes[1,i].imshow(np.clip(img_f,0,1))
        # 1. axes[1,i].set_title('HFlip')
        #axes[1,i].set_title('VFlip')
        #axes[1,i].set_title('Rotation')
        #axes[1,i].set_title('Photometric')
        axes[1,i].set_title('Random Shifts')
        axes[1,i].axis('off')
    #1. plt.suptitle('Patch Augmentation: Horizontal Flip', y=1.02)
    #plt.suptitle('Patch Augmentation: Vertical Flip', y=1.02)
    #plt.suptitle('Patch Augmentation: Rotation', y=1.02)
    #plt.suptitle('Patch Photometric Argumentation', y=1.02)
    plt.suptitle('Patch Random Shifts', y=1.02)
    plt.tight_layout()
    plt.show()

plot_patches()


## 4 · Create Train/Val loaders with HFlip and other arguments

In [ ]:
from sklearn.model_selection import train_test_split

# split by patches (simple split for demo)
dataset_full = PatchDatasetAug(all_rgb, all_mask, PATCH_SIZE, STRIDE, transform=None)
indices = list(range(len(dataset_full)))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=SEED, stratify=[dataset_full[i][1].item() for i in indices])

# train_set = PatchDatasetAug(all_rgb, all_mask, PATCH_SIZE, STRIDE, transform=AUG_HFLIP)
#train_set = PatchDatasetAug(all_rgb, all_mask, PATCH_SIZE, STRIDE, transform=AUG_VFLIP)
#train_set = PatchDatasetAug(all_rgb, all_mask, PATCH_SIZE, STRIDE, transform=AUG_ROT)
#train_set = PatchDatasetAug(all_rgb, all_mask, PATCH_SIZE, STRIDE, transform=AUG_PHOTO)
#train_set = PatchDatasetAug(all_rgb, all_mask, PATCH_SIZE, STRIDE, transform=AUG_ZOOM)
train_set = PatchDatasetAug(all_rgb, all_mask, PATCH_SIZE, STRIDE, transform=AUG_SHIFT)
val_set   = PatchDatasetAug(all_rgb, all_mask, PATCH_SIZE, STRIDE, transform=None)

train_loader = DataLoader(torch.utils.data.Subset(train_set, train_idx), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(torch.utils.data.Subset(val_set, val_idx), batch_size=BATCH_SIZE)

print(f"Train patches: {len(train_idx)}, Val patches: {len(val_idx)}")

## 5 · CNN Model (same as Task 3)

In [ ]:
class VegetationCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU()
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(128, 2)
    
    def forward(self, x):
        x = self.block1(x); x = self.block2(x); x = self.block3(x)
        x = self.gap(x).flatten(1)
        x = self.dropout(x)
        return self.fc(x)

model = VegetationCNN().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()
print(model)

## 6 · Training Loop for Different Types of Arguments 

In [ ]:
# --- CHOOSE AUGMENTATION ---
aug_name = 'random_shift'  # options: 'random_rotation', 'random_brightness', 'random_shift', 'random_flips', 'random_zoom', 'hflip', 'vflip', 'rotation90', 'photometric'
train_transform = AUGMENTATIONS[aug_name]

# rebuild datasets with chosen transform
train_set = PatchDatasetAug(all_rgb, all_mask, PATCH_SIZE, STRIDE, transform=train_transform)
val_set = PatchDatasetAug(all_rgb, all_mask, PATCH_SIZE, STRIDE, transform=None)

from torch.utils.data import DataLoader
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

# reinitialize model for clean comparison
model = VegetationCNN().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = torch.nn.CrossEntropyLoss()

best_val = float('inf')
train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(1, EPOCHS+1):
    # ---- train ----
    model.train()
    running_loss = 0
    running_correct = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * xb.size(0)
        running_correct += (logits.argmax(1) == yb).sum().item()
    
    train_loss = running_loss / len(train_loader.dataset)
    train_acc = running_correct / len(train_loader.dataset)
    
    # ---- validate ----
    model.eval()
    running_loss = 0
    running_correct = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            loss = criterion(logits, yb)
            
            running_loss += loss.item() * xb.size(0)
            running_correct += (logits.argmax(1) == yb).sum().item()
    
    val_loss = running_loss / len(val_loader.dataset)
    val_acc = running_correct / len(val_loader.dataset)
    
    # store
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    
    # save best
    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), f"cnn_{aug_name}.pt")
    if epoch % 5 == 0:
        print(f"Epoch {epoch}/{EPOCHS} - train {train_loss:.4f} ({train_acc:.3f}) - val {val_loss:.4f} ({val_acc:.3f})")

# plot
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(train_losses, label='train'); plt.plot(val_losses, label='val')
plt.legend(); plt.title(f'{aug_name} - Loss')
plt.subplot(1,2,2)
plt.plot(train_accs, label='train'); plt.plot(val_accs, label='val')
plt.legend(); plt.title(f'{aug_name} - Accuracy')
plt.show()


## 7 · Evaluation

In [ ]:
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        preds = model(xb.to(DEVICE)).argmax(1).cpu().numpy()
        y_pred.extend(preds)
        y_true.extend(yb.numpy())

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

results = {
    #"augmentation": "HorizontalFlip",
    #"augmentation": "VerticalFlip",
    #"augmentation": "Rotation",
    #"augmentation": "Photometric Augmentation",
    #"augmentation": "Random Zoom",
    "augmentation": "Random Shift",

    "accuracy": acc,
    "precision": prec,
    "recall": rec,
    "f1": f1
}
print(results)

In [ ]:
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        preds = model(xb.to(DEVICE)).argmax(1).cpu().numpy()
        y_pred.extend(preds); y_true.extend(yb.numpy())

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

#print(f"HFlip Results — Acc: {acc:.4f}, Prec: {prec:.4f}, Rec: {rec:.4f}, F1: {f1:.4f}")
#print(f"VFlip Results — Acc: {acc:.4f}, Prec: {prec:.4f}, Rec: {rec:.4f}, F1: {f1:.4f}")
#print(f"Rotation Results — Acc: {acc:.4f}, Prec: {prec:.4f}, Rec: {rec:.4f}, F1: {f1:.4f}")
#print(f"Photometric Augmentation — Acc: {acc:.4f}, Prec: {prec:.4f}, Rec: {rec:.4f}, F1: {f1:.4f}")
#print(f"Random Zoom — Acc: {acc:.4f}, Prec: {prec:.4f}, Rec: {rec:.4f}, F1: {f1:.4f}")
print(f"Random Shift — Acc: {acc:.4f}, Prec: {prec:.4f}, Rec: {rec:.4f}, F1: {f1:.4f}")


## 8 · Record Results
Copy these numbers into your Task 4 table. Next, duplicate cells 3 and 4 and change AUG_HFLIP to test VFlip, Rotation, etc.

In [ ]:
results = {
    #"augmentation": "HorizontalFlip",
    #"augmentation": "VerticalFlip",
    #"augmentation": "Rotation",
    #"augmentation": "Photometric Augmentation",
    #"augmentation": "Random Zoom",   
    "augmentation": "Random Shift",
    "accuracy": acc,
    "precision": prec,
    "recall": rec,
    "f1": f1
}
print(results)